In [1]:

import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print(module_path)

import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS
from hedging.plot_utils import plot_portfolio_vs_option_price
from hedging.reward_utils import compute_discounted_cumsum_rewards
from hedging.logit_normal import LogitNormal

from torchrl.envs import GymWrapper
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import SafeProbabilisticModule, ProbabilisticActor, TanhNormal


import torch.nn as nn
from tensordict.nn import TensorDictModule, TensorDictSequential
from tensordict import TensorDict

/Users/manu13/Desktop/PHD/DeepHedging/deep_hedging_v0


In [2]:
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 1
transaction_cost = True
transaction_fee_rate = 1e-3

base_env = HedgeCallBS(
    S0, K, maturity, r, sigma, num_paths, num_steps,
    history_len=history_len,
    transaction_cost=transaction_cost,
    transaction_fee_rate=transaction_fee_rate
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env = GymWrapper(base_env, device=device)
env.reset(seed=0)

act_spec = env.specs["input_spec", "full_action_spec", "action"].to(device)

AttributeError: module 'gymnasium.vector' has no attribute 'AutoresetMode'

In [4]:
feat_dim = env.reset()["observation"].shape[-1]

hidden = 128
action_dim = 1  
inact_dim = 1   

In [5]:
class LatestStateAction(nn.Module):
    def forward(self, obs):  
        if obs.dim() == 3:          
            latest = obs[:, -1, :]
        else:                       
            latest = obs
        return latest
                  
class LatestStateInaction(nn.Module):
    def forward(self, obs):  
        if obs.dim() == 3:          
            latest = obs[:, -1, :]
        else:                       
            latest = obs
        return latest               

In [6]:
class PolicyHead(nn.Module):
    def __init__(self, in_dim, hidden_size, action_dim=1, min_log_std=1e-3, max_log_std=2):
        super().__init__()
        self.mu_net = nn.Sequential(
            nn.Linear(in_dim, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, action_dim),
            nn.Tanh()
        )

        self.std_net = nn.Sequential(
            nn.Linear(in_dim, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, action_dim),
        )

        self.log_min = min_log_std
        self.log_max = max_log_std

    def forward(self, x):
        mu = self.mu_net(x) # x = x[:, -1, :] ??
        log_std = self.std_net(x)
        log_std = torch.clamp(log_std, self.log_min, self.log_max)
        return mu, log_std


class InactionHead(nn.Module):
    def __init__(self, in_dim, hidden_size, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden, out_dim)  # logits
        )

    def forward(self, x):
        return self.net(x)

latest_state_actor = TensorDictModule(
    module=LatestStateAction(),
    in_keys=["observation"],
    out_keys=["latest_state"],
)

latest_state_inactor = TensorDictModule(
    module=LatestStateInaction(),
    in_keys=["observation"],
    out_keys=["latest_state"],
)

policy_head = TensorDictModule(
    module=PolicyHead(feat_dim, hidden, action_dim),
    in_keys=["latest_state"],
    out_keys=["loc", "scale"],
)

inaction_head = TensorDictModule(
    module=InactionHead(feat_dim, hidden, inact_dim),
    in_keys=["latest_state"],
    out_keys=["logits"],
)

In [7]:
action_spec = env.specs["input_spec", "full_action_spec", "action"].to(device)

In [8]:
module_actor = ProbabilisticActor(
    module=policy_head,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    spec=action_spec,
    distribution_class=LogitNormal,
    return_log_prob=True,
    log_prob_key="action_log_prob",
)

module_inactor = ProbabilisticActor(
    module=inaction_head,
    in_keys=["logits"],
    out_keys=["inact"],
    spec=action_spec,
    distribution_class=torch.distributions.Bernoulli,
    return_log_prob=True,
    log_prob_key="inaction_log_prob",
)

In [9]:
actor = TensorDictSequential(
    latest_state_actor,
    module_actor,
).to(device)

inactor = TensorDictSequential(
    latest_state_inactor,
    module_inactor,
).to(device)

In [10]:
from torch.optim.lr_scheduler import CosineAnnealingLR

lr = 5e-4 
optimizer_actor = torch.optim.Adam(actor.parameters(), lr=lr)
optimizer_inactor = torch.optim.Adam(inactor.parameters(), lr=lr)

num_epochs = 20
num_episodes = 200
policy_only_epochs = 8
joint_training_epochs = num_epochs - policy_only_epochs
gamma = 0.999

total_steps = num_epochs * num_episodes

scheduler_actor = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_actor, T_max=total_steps, eta_min=1e-5
)
scheduler_inactor = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_inactor, T_max=total_steps, eta_min=1e-5
)

In [11]:
print("=== CURRICULUM LEARNING (TorchRL) WITH TRANSACTION COSTS ===")
print(f"Phase 1: Policy-only for {policy_only_epochs} epochs")
print(f"Phase 2: Joint (policy + inaction) for {joint_training_epochs} epochs")

=== CURRICULUM LEARNING (TorchRL) WITH TRANSACTION COSTS ===
Phase 1: Policy-only for 8 epochs
Phase 2: Joint (policy + inaction) for 12 epochs


In [11]:

with set_exploration_type(ExplorationType.RANDOM):
    for epoch in range(num_epochs):
        
        if epoch < policy_only_epochs:
            print(f"\n Epoch: {epoch+1}/{num_epochs} - PHASE 1: Policy Network Training Only")

            for episode in range(num_episodes):
                td_episode = env.reset()
                
                td_episode = env.rollout(
                    policy=actor,
                    auto_reset=True,
                    auto_cast_to_device=True,
                    break_when_all_done=True,
                    max_steps=num_steps,
                )

                rewards = td_episode["next", "reward"].squeeze(-1)

                R = compute_discounted_cumsum_rewards(np.array(rewards), gamma)
                R = R - R.mean(axis=1, keepdims=True)
                R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
                R = torch.tensor(R, dtype=torch.float32).to(device)

                action_log_probs = td_episode.get("action_log_prob")

                optimizer_actor.zero_grad()
                actor_loss = (-R * action_log_probs).mean() # torch.stack(action_log_probs)?
                actor_loss.backward()
                optimizer_actor.step()
                

                if (episode + 1) % 20 == 0:
                    print(
                        f"  Episode {episode + 1}/{num_episodes}, "
                        f"Policy Loss: {actor_loss.item():.4f}, "
                        f"Avg. Reward: {np.array(rewards).mean():.4f}"
                    )
                    scheduler_actor.step()
        
        else:
            print(f"\n Epoch: {epoch+1}/{num_epochs} - PHASE 2: Joint Policy Training")

            for episode in range(num_episodes):
                td_episode = env.reset()
                prev_action = torch.zeros(env.batch_size[0], action_dim, device=device)
                log_probs, inact_log_probs, rewards = [], [], []

                for t in range(num_steps):
                    td_episode = actor(td_episode)    
                    td_episode = inactor(td_episode)  

                    mask = td_episode["inact"].squeeze(-1).squeeze(-1).bool().expand_as(td_episode["action"])
                    final_action = torch.where(mask, prev_action.squeeze(-1), td_episode["action"])
                    chosen_logp_inact = td_episode["inaction_log_prob"]
                    logp_action_t = td_episode["action_log_prob"]

                    td_episode = td_episode.clone(True)
                    td_episode.set_("action", final_action.detach())
                    td_episode = env.step(td_episode)

                    rewards.append(td_episode["next", "reward"])             
                    log_probs.append(logp_action_t)                  
                    inact_log_probs.append(chosen_logp_inact)        

                    prev_action = final_action.detach()
                
                rewards_t = torch.stack(rewards, dim=0)                        
                rewards_np = rewards_t.cpu().numpy().squeeze(-1)
                
                R = compute_discounted_cumsum_rewards(np.array(rewards_np), gamma)
                R = R - R.mean(axis=1, keepdims=True)
                R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
                R = torch.tensor(R, dtype=torch.float32, device=device)        

                log_probs_t = torch.stack(log_probs, dim=0)                    
                inact_log_probs_t = torch.stack(inact_log_probs, dim=0)       

               
                optimizer_inactor.zero_grad()
                inact_loss = (-R * inact_log_probs_t.squeeze(-1)).mean()
                inact_loss.backward()
                optimizer_inactor.step()
                

                if episode % 3 == 0:
                    optimizer_actor.zero_grad()
                    actor_loss = (-R * log_probs_t).mean()
                    actor_loss.backward()
                    optimizer_actor.step()
                    current_policy_loss = actor_loss.item()
                else:
                    current_policy_loss = 0.0
                
                if (episode + 1) % 20 == 0:
                    policy_info = f"Policy Loss: {current_policy_loss:.4f}" if episode % 3 == 0 else "Policy: No Update"
                    print(
                        f"  Episode {episode + 1}/{num_episodes}, "
                        f"{policy_info}, "
                        f"Inaction Loss: {inact_loss.item():.4f}, "
                        f"Avg. Reward: {rewards_t.mean().item():.4f}"
                    )
                    scheduler_inactor.step()


 Epoch: 1/20 - PHASE 1: Policy Network Training Only


/var/folders/j0/tqzlnmks6cx221q8pl95104w0000gn/T/ipykernel_10930/554093960.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  R = compute_discounted_cumsum_rewards(np.array(rewards), gamma)
/var/folders/j0/tqzlnmks6cx221q8pl95104w0000gn/T/ipykernel_10930/554093960.py:37: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  f"Avg. Reward: {np.array(rewards).mean():.4f}"


  Episode 20/200, Policy Loss: -0.5528, Avg. Reward: -3.5002
  Episode 40/200, Policy Loss: -1.5362, Avg. Reward: -3.6245
  Episode 60/200, Policy Loss: -1.9492, Avg. Reward: -3.3243
  Episode 80/200, Policy Loss: -2.1576, Avg. Reward: -3.7267
  Episode 100/200, Policy Loss: -2.2339, Avg. Reward: -3.7315
  Episode 120/200, Policy Loss: -2.2452, Avg. Reward: -3.5913
  Episode 140/200, Policy Loss: -2.3318, Avg. Reward: -3.3479
  Episode 160/200, Policy Loss: -2.3873, Avg. Reward: -3.5689
  Episode 180/200, Policy Loss: -2.4077, Avg. Reward: -3.7373
  Episode 200/200, Policy Loss: -2.4055, Avg. Reward: -3.6388

 Epoch: 2/20 - PHASE 1: Policy Network Training Only
  Episode 20/200, Policy Loss: -2.4714, Avg. Reward: -3.7051
  Episode 40/200, Policy Loss: -2.4160, Avg. Reward: -3.5581
  Episode 60/200, Policy Loss: -2.5118, Avg. Reward: -3.5705
  Episode 80/200, Policy Loss: -2.4992, Avg. Reward: -3.5471
  Episode 100/200, Policy Loss: -2.5768, Avg. Reward: -3.6807
  Episode 120/200, Polic

In [17]:
class TestPolicy(nn.Module):
    def __init__(self, actor, inactor):
        super().__init__()
        self.actor = actor
        self.inactor = inactor

    def forward(self, td):
        # Deterministic action from the actor
        with torch.no_grad():
            td = self.actor(td)
            proposed_action = td["action"]  
        
        td = self.inactor(td)
        mask = td["inact"].squeeze(-1).bool().expand_as(proposed_action)

        prev_action = td.get("prev_action", torch.zeros_like(proposed_action))
        final_action = torch.where(mask, prev_action, proposed_action)

        td.set("action", final_action)
        td.set("prev_action", final_action.detach())  # save for next step

        return td


In [18]:
base_env = HedgeCallBS(S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

TensorDict(
    fields={
        done: Tensor(shape=torch.Size([30, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        observation: Tensor(shape=torch.Size([30, 1, 11]), device=cpu, dtype=torch.float32, is_shared=False),
        terminated: Tensor(shape=torch.Size([30, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([30, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([30]),
    device=cpu,
    is_shared=False)

In [19]:
test_policy = TestPolicy(actor, inactor)

with torch.no_grad(), set_exploration_type(ExplorationType.RANDOM):
    td_rollout = env.rollout(
        max_steps=num_steps,
        policy=test_policy,
        auto_reset=True,
        break_when_all_done=True,
    )

print("Test rollout complete!")
print("Average Reward:", td_rollout["next", "reward"].mean().item())


Test rollout complete!
Average Reward: -4.705020904541016


In [20]:
plot_portfolio_vs_option_price(env._env)